In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print("CUDA available:", torch.cuda.is_available())
print("Num GPUs:", torch.cuda.device_count())
assert torch.cuda.is_available(), "Không thấy GPU! Vào Settings > Accelerator, bật GPU rồi chạy lại."


In [ ]:
%cd /kaggle/working
!rm -rf SEMamba
!git clone --depth 1 https://github.com/RoyChao19477/SEMamba.git
%cd /kaggle/working/SEMamba
!ls


In [ ]:
import os

for pkg_dir in ["utils", "models", "dataloaders"]:
    init_file = os.path.join(pkg_dir, "__init__.py")
    if not os.path.exists(init_file):
        open(init_file, "w").close()
        print(f"Da tao {init_file}")
    else:
        print(f"{init_file} da ton tai")


In [ ]:
!pip install -q packaging librosa soundfile pyyaml tensorboard pesq einops joblib triton ninja


In [ ]:
import subprocess
r = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
print(r.stdout, r.stderr)
print("nvcc co san:" , r.returncode == 0)


In [ ]:
import subprocess, torch

print("torch:", torch.__version__, "| cuda:", torch.version.cuda)

def run_show(cmd, cwd=None, env=None):
    print("$", " ".join(cmd))
    p = subprocess.run(cmd, cwd=cwd, env=env, capture_output=True, text=True)
    print(p.stdout[-4000:])
    if p.returncode != 0:
        print(p.stderr[-4000:])
    print("Return code:", p.returncode)
    return p.returncode

rc = run_show(["pip", "install", "mamba-ssm", "causal-conv1d", "--no-build-isolation"])

if rc == 0:
    import importlib
    import mamba_ssm
    importlib.reload(mamba_ssm)
    print("mamba_ssm da san sang (wheel prebuilt tu GitHub Releases).")
else:
    print("Van khong duoc. Copy toan bo log stderr o tren gui lai de debug tiep "
          "(tim dong co chu 'error:', 'nvcc', 'CUDA_HOME', hoac 'No matching distribution').")


In [ ]:
import importlib
import mamba_ssm, torch, einops, librosa, soundfile, pesq
importlib.reload(mamba_ssm)
print("mamba_ssm OK, torch", torch.__version__, "| CUDA build:", torch.version.cuda)
print("mamba_ssm version:", getattr(mamba_ssm, '__version__', 'khong ro'))


In [ ]:
mamba_block_patched = r'''# Reference: https://github.com/state-spaces/mamba/blob/9127d1f47f367f5c9cc49c73ad73557089d02cb8/mamba_ssm/models/mixer_seq_simple.py
# NOTE: da patch de tuong thich ca mamba_ssm <=1.2.x va >=2.x

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import init
from torch.nn.parameter import Parameter
from functools import partial
from einops import rearrange

from mamba_ssm.modules.mamba_simple import Mamba

try:
    # mamba_ssm <= 1.2.x: Block nam trong mamba_simple, khong can mlp_cls
    from mamba_ssm.modules.mamba_simple import Block
    _NEEDS_MLP_CLS = False
except ImportError:
    # mamba_ssm >= 2.x: Block chuyen sang modules.block, bat buoc truyen mlp_cls
    from mamba_ssm.modules.block import Block
    _NEEDS_MLP_CLS = True

from mamba_ssm.models.mixer_seq_simple import _init_weights

try:
    from mamba_ssm.ops.triton.layernorm import RMSNorm
except ImportError:
    from mamba_ssm.ops.triton.layer_norm import RMSNorm

def create_block(
    d_model, cfg, layer_idx=0, rms_norm=True, fused_add_norm=False, residual_in_fp32=False,
    ):
    d_state = cfg['model_cfg']['d_state']
    d_conv = cfg['model_cfg']['d_conv']
    expand = cfg['model_cfg']['expand']
    norm_epsilon = cfg['model_cfg']['norm_epsilon']

    mixer_cls = partial(Mamba, layer_idx=layer_idx, d_state=d_state, d_conv=d_conv, expand=expand)
    norm_cls = partial(
        nn.LayerNorm if not rms_norm else RMSNorm, eps=norm_epsilon
    )
    block_kwargs = {'mlp_cls': nn.Identity} if _NEEDS_MLP_CLS else {}
    block = Block(
            d_model,
            mixer_cls,
            norm_cls=norm_cls,
            fused_add_norm=fused_add_norm,
            residual_in_fp32=residual_in_fp32,
            **block_kwargs,
            )
    block.layer_idx = layer_idx
    return block

class MambaBlock(nn.Module):
    def __init__(self, in_channels, cfg):
        super(MambaBlock, self).__init__()
        n_layer = 1
        self.forward_blocks  = nn.ModuleList( create_block(in_channels, cfg) for i in range(n_layer) )
        self.backward_blocks = nn.ModuleList( create_block(in_channels, cfg) for i in range(n_layer) )

        self.apply(
            partial(
                _init_weights,
                n_layer=n_layer,
            )
        )

    def forward(self, x):
        x_forward, x_backward = x.clone(), torch.flip(x, [1])
        resi_forward, resi_backward = None, None

        for layer in self.forward_blocks:
            x_forward, resi_forward = layer(x_forward, resi_forward)
        y_forward = (x_forward + resi_forward) if resi_forward is not None else x_forward

        for layer in self.backward_blocks:
            x_backward, resi_backward = layer(x_backward, resi_backward)
        y_backward = torch.flip((x_backward + resi_backward), [1]) if resi_backward is not None else torch.flip(x_backward, [1])

        return torch.cat([y_forward, y_backward], -1)

class TFMambaBlock(nn.Module):
    # Temporal-Frequency Mamba block for sequence modeling.
    def __init__(self, cfg):
        super(TFMambaBlock, self).__init__()
        self.cfg = cfg
        self.hid_feature = cfg['model_cfg']['hid_feature']

        self.time_mamba = MambaBlock(in_channels=self.hid_feature, cfg=cfg)
        self.freq_mamba = MambaBlock(in_channels=self.hid_feature, cfg=cfg)

        self.tlinear = nn.ConvTranspose1d(self.hid_feature * 2, self.hid_feature, 1, stride=1)
        self.flinear = nn.ConvTranspose1d(self.hid_feature * 2, self.hid_feature, 1, stride=1)

    def forward(self, x):
        b, c, t, f = x.size()

        x = x.permute(0, 3, 2, 1).contiguous().view(b*f, t, c)
        x = self.tlinear( self.time_mamba(x).permute(0,2,1) ).permute(0,2,1) + x
        x = x.view(b, f, t, c).permute(0, 2, 1, 3).contiguous().view(b*t, f, c)
        x = self.flinear( self.freq_mamba(x).permute(0,2,1) ).permute(0,2,1) + x
        x = x.view(b, t, f, c).permute(0, 3, 1, 2)
        return x
'''

with open("models/mamba_block.py", "w") as f:
    f.write(mamba_block_patched)

print("Da ghi de models/mamba_block.py voi ban tuong thich ca 2 API cua mamba_ssm.")


In [ ]:
import importlib, sys

for mod in list(sys.modules):
    if mod.startswith("models"):
        del sys.modules[mod]

from models.generator import SEMamba
from models.discriminator import MetricDiscriminator
print("Import models.generator / models.discriminator THANH CONG. Patch hoat dong dung.")


In [ ]:
disc_path = "models/discriminator.py"
with open(disc_path, "r") as f:
    disc_src = f.read()

old = "def batch_pesq(clean, noisy, cfg):\n    num_worker = cfg['env_setting']['num_workers']\n    pesq_score = Parallel(n_jobs=num_worker)(delayed(pesq_loss)(c, n) for c, n in zip(clean, noisy))"
new = "def batch_pesq(clean, noisy, cfg):\n    # SUA: khong dung cfg['env_setting']['num_workers'] nua (gia tri nay = 0 de\n    # tranh deadlock DataLoader, nhung joblib khong chap nhan n_jobs=0).\n    # Chay tuan tu (n_jobs=1) de an toan, khong fork them tien trinh.\n    pesq_score = Parallel(n_jobs=1)(delayed(pesq_loss)(c, n) for c, n in zip(clean, noisy))"

assert old in disc_src, "Khong tim thay doan can patch, kiem tra lai file goc."
disc_src = disc_src.replace(old, new)

with open(disc_path, "w") as f:
    f.write(disc_src)

print("Da patch batch_pesq trong models/discriminator.py de dung n_jobs=1 co dinh.")


In [ ]:
import os, glob

TRAIN_CLEAN_DIR = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/CLEAN"
TRAIN_NOISY_DIR = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/NOISE"
TEST_DIR        = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TEST "  # co dau cach sau "TEST"

paths = {"TRAIN_CLEAN_DIR": TRAIN_CLEAN_DIR, "TRAIN_NOISY_DIR": TRAIN_NOISY_DIR, "TEST_DIR": TEST_DIR}
missing = []
for name, p in paths.items():
    ok = os.path.isdir(p)
    print(f"{name}: {'OK' if ok else 'KHONG TIM THAY'}  ->  {p!r}")
    if not ok:
        missing.append(name)

if missing:
    print("\nMot so duong dan khong ton tai. Dang tim tu dong trong /kaggle/input ...")
    candidates = glob.glob("/kaggle/input/**/NOISE SPEECH", recursive=True)
    print("Cac thu muc 'NOISE SPEECH' tim thay:")
    for c in candidates:
        print(" -", repr(c))
    print("\n=> Neu thay duong dan dung o tren, hay copy va gan lai vao TRAIN_CLEAN_DIR / TRAIN_NOISY_DIR / TEST_DIR roi chay lai cell nay.")


In [ ]:
def list_wavs(directory):
    files = []
    for root, dirs, filenames in os.walk(directory):
        for fn in filenames:
            if fn.lower().endswith('.wav'):
                files.append(os.path.join(root, fn))
    return files

for name, path in [("TRAIN/CLEAN", TRAIN_CLEAN_DIR), ("TRAIN/NOISE", TRAIN_NOISY_DIR), ("TEST", TEST_DIR)]:
    print(f"--- {name} ({path}) ---")
    if not os.path.isdir(path):
        print("  (khong ton tai, bo qua)\n")
        continue
    sub_entries = os.listdir(path)
    print("  Entries cap 1 (toi da 10):", sub_entries[:10])
    wavs = list_wavs(path)
    print("  Tong so file .wav (bao gom thu muc con):", len(wavs))
    print("  Vi du ten file:", [os.path.basename(w) for w in wavs[:5]])
    print()


In [ ]:
import json, random

clean_files = list_wavs(TRAIN_CLEAN_DIR)
noisy_files = list_wavs(TRAIN_NOISY_DIR)
print(f"CLEAN: {len(clean_files)} file | NOISE: {len(noisy_files)} file")

clean_by_name = {os.path.basename(f): f for f in clean_files}
noisy_by_name = {os.path.basename(f): f for f in noisy_files}

common = sorted(set(clean_by_name) & set(noisy_by_name))
only_noisy = sorted(set(noisy_by_name) - set(clean_by_name))
only_clean = sorted(set(clean_by_name) - set(noisy_by_name))

print(f"So cap ten file KHOP giua CLEAN va NOISE: {len(common)}")
if only_noisy:
    print(f"CANH BAO: {len(only_noisy)} file trong NOISE khong co ban CLEAN cung ten. VD: {only_noisy[:5]}")
if only_clean:
    print(f"CANH BAO: {len(only_clean)} file trong CLEAN khong co ban NOISE cung ten. VD: {only_clean[:5]}")

assert len(common) > 0, (
    "Khong tim thay cap file CLEAN/NOISE nao trung ten! "
    "Kiem tra lai: 2 thu muc phai chua file .wav CUNG TEN cho cung 1 cau noi (1 ban sach, 1 ban co nhieu)."
)


In [ ]:
random.seed(1234)
common_shuffled = common[:]
random.shuffle(common_shuffled)

VALID_RATIO = 0.05  
n_valid = max(1, int(len(common_shuffled) * VALID_RATIO))
valid_names = common_shuffled[:n_valid]
train_names = common_shuffled[n_valid:]

print(f"Train: {len(train_names)} cap | Valid: {len(valid_names)} cap")

def dump_json(name_to_path, names, out_path):
    paths_out = [name_to_path[n] for n in names]
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, 'w') as f:
        json.dump(paths_out, f, indent=4)
    print(f"Da ghi {len(paths_out)} duong dan vao {out_path}")

dump_json(clean_by_name, train_names, "data/train_clean.json")
dump_json(noisy_by_name, train_names, "data/train_noisy.json")
dump_json(clean_by_name, valid_names, "data/valid_clean.json")
dump_json(noisy_by_name, valid_names, "data/valid_noisy.json")


In [ ]:
import shutil

test_wavs = list_wavs(TEST_DIR)
print(f"So file .wav trong TEST: {len(test_wavs)}")

subdirs = [d for d in os.listdir(TEST_DIR) if os.path.isdir(os.path.join(TEST_DIR, d))] if os.path.isdir(TEST_DIR) else []
print("Thu muc con trong TEST:", subdirs)

clean_sub, noisy_sub = None, None
for d in subdirs:
    dl = d.lower()
    if 'clean' in dl and clean_sub is None:
        clean_sub = os.path.join(TEST_DIR, d)
    if ('noise' in dl or 'noisy' in dl) and noisy_sub is None:
        noisy_sub = os.path.join(TEST_DIR, d)

if clean_sub and noisy_sub:
    print(f"Phat hien cau truc TEST/CLEAN ({clean_sub}) + TEST/NOISE ({noisy_sub})")
    tc_by_name = {os.path.basename(f): f for f in list_wavs(clean_sub)}
    tn_by_name = {os.path.basename(f): f for f in list_wavs(noisy_sub)}
    common_test = sorted(set(tc_by_name) & set(tn_by_name))
    print(f"So cap test khop ten: {len(common_test)}")
    if common_test:
        dump_json(tc_by_name, common_test, "data/test_clean.json")
        dump_json(tn_by_name, common_test, "data/test_noisy.json")
    else:
        print("Khong co cap nao khop ten trong TEST, fallback sang dung valid set.")
        shutil.copy("data/valid_clean.json", "data/test_clean.json")
        shutil.copy("data/valid_noisy.json", "data/test_noisy.json")
else:
    print("TEST khong co cau truc CLEAN/NOISE ro rang.")
    print("-> Dung tap valid o Buoc 5 lam test_clean.json/test_noisy.json (de train.py doc config khong loi).")
    shutil.copy("data/valid_clean.json", "data/test_clean.json")
    shutil.copy("data/valid_noisy.json", "data/test_noisy.json")
    print(f"-> {len(test_wavs)} file trong TEST se duoc dung de chay inference/enhance truc tiep o Buoc 9.")


In [ ]:
import yaml, torch

with open("recipes/SEMamba_advanced/SEMamba_advanced.yaml") as f:
    cfg = yaml.safe_load(f)

num_gpus = max(1, torch.cuda.device_count())

cfg['env_setting']['num_gpus'] = num_gpus

cfg['env_setting']['num_workers'] = 0           
cfg['env_setting']['checkpoint_interval'] = 200
cfg['env_setting']['validation_interval'] = 200
cfg['env_setting']['summary_interval'] = 20
cfg['env_setting']['stdout_interval'] = 10
cfg['env_setting']['dist_cfg']['dist_url'] = 'tcp://localhost:19478'

cfg['training_cfg']['training_epochs'] = 20     
cfg['training_cfg']['batch_size'] = 4
cfg['training_cfg']['learning_rate'] = 0.00005  
cfg['training_cfg']['use_PCS400'] = False

EXP_NAME = "SEMamba_finetune"
os.makedirs("recipes/SEMamba_finetune", exist_ok=True)
CONFIG_PATH = "recipes/SEMamba_finetune/finetune.yaml"
with open(CONFIG_PATH, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"So GPU phat hien: {num_gpus}")
print(yaml.dump(cfg, sort_keys=False))


In [ ]:
import torch
from models.generator import SEMamba
from models.discriminator import MetricDiscriminator
from utils.util import load_config

cfg = load_config(CONFIG_PATH)
EXP_PATH = f"exp/{EXP_NAME}"
os.makedirs(EXP_PATH, exist_ok=True)

device = torch.device("cuda:0")
generator = SEMamba(cfg).to(device)
discriminator = MetricDiscriminator().to(device)

# Nap trong so pretrained cho generator
g_ckpt = torch.load("ckpts/SEMamba_advanced.pth", map_location=device)
missing, unexpected = generator.load_state_dict(g_ckpt['generator'], strict=False)
print("Da nap generator pretrained. missing keys:", len(missing), "| unexpected keys:", len(unexpected))


try:
    d_state = torch.load("ckpts/pretrained_discriminator.pth", map_location=device)
    discriminator.load_state_dict(d_state, strict=False)
    print("Da nap discriminator pretrained tu ckpts/pretrained_discriminator.pth")
except Exception as e:
    print("Khong nap duoc discriminator pretrained (se train discriminator tu dau):", e)

lr = cfg['training_cfg']['learning_rate']
betas = (cfg['training_cfg']['adam_b1'], cfg['training_cfg']['adam_b2'])
optim_g = torch.optim.AdamW(generator.parameters(), lr=lr, betas=betas)
optim_d = torch.optim.AdamW(discriminator.parameters(), lr=lr, betas=betas)

torch.save({'generator': generator.state_dict()}, f"{EXP_PATH}/g_00000000.pth")
torch.save({
    'discriminator': discriminator.state_dict(),
    'optim_g': optim_g.state_dict(),
    'optim_d': optim_d.state_dict(),
    'steps': 0,
    'epoch': -1,
}, f"{EXP_PATH}/do_00000000.pth")

print("Da tao checkpoint khoi dau cho finetune tai:", EXP_PATH)
del generator, discriminator, optim_g, optim_d
torch.cuda.empty_cache()


In [ ]:
import subprocess, os

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["PYTHONPATH"] = "/kaggle/working/SEMamba" + os.pathsep + env.get("PYTHONPATH", "")

cmd = [
    "python", "train.py",
    "--config", CONFIG_PATH,
    "--exp_folder", "exp",
    "--exp_name", EXP_NAME,
]
print("Chay:", " ".join(cmd))

proc = subprocess.Popen(cmd, env=env, cwd="/kaggle/working/SEMamba",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("\nKet thuc voi return code:", proc.returncode)


In [ ]:
import glob

ckpts = sorted(glob.glob(f"{EXP_PATH}/g_*.pth"))
assert ckpts, "Chua co checkpoint nao duoc luu (chua du 1 checkpoint_interval step). Kiem tra lai buoc train o tren."
latest_ckpt = ckpts[-1]
print("Se dung checkpoint:", latest_ckpt)


In [ ]:
import subprocess, os

OUTPUT_DIR = "/kaggle/working/enhanced_test"
os.makedirs(OUTPUT_DIR, exist_ok=True)

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["PYTHONPATH"] = "/kaggle/working/SEMamba" + os.pathsep + env.get("PYTHONPATH", "")

cmd = [
    "python", "inference.py",
    "--input_folder", TEST_DIR,     
    "--output_folder", OUTPUT_DIR,
    "--checkpoint_file", latest_ckpt,
    "--config", CONFIG_PATH,
    "--post_processing_PCS", "False",
]
print("Chay:", cmd)

proc = subprocess.Popen(cmd, env=env, cwd="/kaggle/working/SEMamba",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("\nKet thuc voi return code:", proc.returncode)
print("File enhance da luu tai:", OUTPUT_DIR)
